Установка

In [1]:
!pip install -q langchain langchain-openai langgraph langfuse rank-bm25 faiss-cpu

print("✅ Установлено")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
✅ Установлено


Импорты и секреты

In [2]:
import os, time, json, numpy as np, faiss
from google.colab import userdata
from openai import OpenAI
from rank_bm25 import BM25Okapi
from langfuse import Langfuse
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

# OpenAI
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# LangFuse
langfuse = Langfuse(
    public_key=userdata.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=userdata.get("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com"
)

print("✅ Готово")

✅ Готово


База знаний

In [3]:
docs = [
    {"title": "Расторжение аренды", "text": "Расторжение договора аренды возможно за 30 дней. Нужно письменное уведомление. Депозит возвращается в течение 10 рабочих дней."},
    {"title": "Арендная ставка", "text": "Базовая ставка 1200 руб/м² в месяц. Коэффициент этажа: 1 этаж — 1.2, 2-10 — 1.0, выше 10 — 1.1. Коэффициент площади: до 50м² — 1.3, 50-100м² — 1.0, более 100м² — 0.8."},
    {"title": "Приёмка помещения", "text": "Проверить: электрику, сантехнику, окна, двери, стены. Акт приёмки в двух экземплярах."},
    {"title": "Парковка", "text": "Одно место включено для помещений более 50м². Дополнительное место — 3500 руб/мес. Гостевые — бесплатно до 2 часов."},
    {"title": "Штрафы", "text": "Просрочка аренды — пеня 0.1% в день. Порча имущества — стоимость + 20% штраф. Досрочное расторжение без уведомления — депозит не возвращается."},
    {"title": "Страхование", "text": "Страхование обязательно: от пожара, затопления, кражи. Сумма — не менее годовой арендной платы. Полис за 5 дней."},
    {"title": "Оплата аренды", "text": "Оплата до 5 числа каждого месяца. Способы: банковский перевод, личный кабинет, наличные. Указывать номер договора."},
    {"title": "График работы", "text": "Офис: Пн-Пт 9:00-20:00, Сб 10:00-16:00, Вс — выходной. Телефон: +7 (831) 555-00-01."},
    {"title": "Ремонт", "text": "Перепланировка — только с согласования. Косметический ремонт — за счёт арендатора. Капитальный — за счёт арендодателя. После ремонта — фотоотчёт."},
    {"title": "Рабочие места", "text": "Включены в аренду: стол, стул, шкаф. Дополнительная мебель — по запросу, стоимость согласовывается отдельно."}
]

texts = [d["text"] for d in docs]
print(f"✅ База: {len(docs)} документов")

✅ База: 10 документов


Эмбеддинги и индексы

In [4]:
def get_embedding(text):
    r = client.embeddings.create(model="text-embedding-3-small", input=text)
    return np.array(r.data[0].embedding)

print("⏳ Эмбеддинги...")
embeddings = np.array([get_embedding(t) for t in texts])

# FAISS
dim = embeddings.shape[1]
faiss_idx = faiss.IndexFlatIP(dim)
faiss_idx.add(embeddings.astype('float32'))

# BM25
tokenized = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized)

print(f"✅ FAISS: {faiss_idx.ntotal} | BM25: готов | dim={dim}")

⏳ Эмбеддинги...
✅ FAISS: 10 | BM25: готов | dim=1536


Гибридный поиск

In [5]:
def hybrid_search(query, k=3):
    q_vec = get_embedding(query)

    # Dense
    d_scores, d_idx = faiss_idx.search(q_vec.reshape(1,-1).astype('float32'), 10)

    # Sparse
    s_scores = bm25.get_scores(query.lower().split())
    s_idx = s_scores.argsort()[-10:][::-1]

    # Объединение 0.7/0.3
    scores = {}
    for i, idx in enumerate(d_idx[0]):
        scores[idx] = scores.get(idx, 0) + 0.7 * float(d_scores[0][i])
    for idx in s_idx:
        if s_scores[idx] > 0:
            scores[idx] = scores.get(idx, 0) + 0.3 * float(s_scores[idx])

    top = sorted(scores, key=scores.get, reverse=True)[:k]
    return [{"text": texts[i], "score": scores[i], "title": docs[i]["title"]} for i in top]

In [10]:
def hybrid_search(query, k=3):
    q_vec = get_embedding(query)

    # Dense (FAISS)
    d_scores, d_idx = faiss_idx.search(q_vec.reshape(1,-1).astype('float32'), 10)

    # Нормализация dense
    d_max = d_scores[0].max() if d_scores[0].max() > 0 else 1.0
    d_norm = {idx: float(d_scores[0][i]) / d_max for i, idx in enumerate(d_idx[0])}

    # Sparse (BM25)
    s_scores = bm25.get_scores(query.lower().split())
    s_idx = s_scores.argsort()[-10:][::-1]
    s_max = s_scores.max() if s_scores.max() > 0 else 1.0
    s_norm = {idx: float(s_scores[idx]) / s_max for idx in s_idx if s_scores[idx] > 0}

    # Гибрид 0.7/0.3
    all_idx = set(list(d_norm.keys()) + list(s_norm.keys()))
    scores = {}
    for idx in all_idx:
        scores[idx] = 0.7 * d_norm.get(idx, 0) + 0.3 * s_norm.get(idx, 0)

    top = sorted(scores, key=scores.get, reverse=True)[:k]
    return [{"text": texts[i], "score": round(scores[i], 2), "title": docs[i]["title"]} for i in top]

# Тест
for q in ["расторжение договора", "парковка", "еда"]:
    r = hybrid_search(q)
    print(f"🔍 {q}")
    for item in r:
        print(f"   [{item['score']:.2f}] {item['title']}")
    print()

🔍 расторжение договора
   [1.00] Расторжение аренды
   [0.66] Штрафы
   [0.45] Оплата аренды

🔍 парковка
   [0.70] Парковка
   [0.64] Штрафы
   [0.64] Страхование

🔍 еда
   [0.70] Приёмка помещения
   [0.49] Штрафы
   [0.47] Рабочие места



LangGraph-агент

In [11]:
class State(TypedDict):
    query: str
    chunks: List[dict]
    answer: str
    confidence: float
    escalated: bool

def retrieve(state: State):
    chunks = hybrid_search(state["query"])
    state["chunks"] = chunks
    state["confidence"] = sum(c["score"] for c in chunks) / len(chunks) if chunks else 0
    return state

def validate(state: State):
    state["escalated"] = state["confidence"] < 0.5 or not state["chunks"]
    return state

def answer(state: State):
    context = "\n\n".join(c["text"] for c in state["chunks"])
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Ты — ассистент агентства недвижимости. Отвечай только по контексту. Кратко, по делу, с цифрами."},
            {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {state['query']}"}
        ],
        temperature=0
    )
    state["answer"] = r.choices[0].message.content
    return state

def escalate(state: State):
    state["answer"] = "Информация не найдена. Ваш вопрос передан оператору. Мы свяжемся с вами в ближайшее время."
    state["escalated"] = True
    return state

def route(state: State):
    return "escalate" if state["escalated"] else "answer"

# Граф
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("validate", validate)
g.add_node("answer", answer)
g.add_node("escalate", escalate)

g.set_entry_point("retrieve")
g.add_edge("retrieve", "validate")
g.add_conditional_edges("validate", route, {"answer": "answer", "escalate": "escalate"})
g.add_edge("answer", END)
g.add_edge("escalate", END)

agent = g.compile()
print("✅ LangGraph-агент готов")

✅ LangGraph-агент готов


Тест агента

In [12]:
for q in ["Как расторгнуть договор?", "Сколько стоит парковка?", "Где покушать?"]:
    t0 = time.time()

    result = agent.invoke({"query": q, "chunks": [], "answer": "", "confidence": 0.0, "escalated": False})

    latency = round((time.time() - t0) * 1000, 2)

    print(f"❓ {q}")
    print(f"🤖 {result['answer'][:150]}...")
    print(f"📊 confidence={result['confidence']:.2f} | escalated={result['escalated']} | latency={latency}ms")
    print("---")

❓ Как расторгнуть договор?
🤖 Для расторжения договора аренды необходимо:

1. Подготовить письменное уведомление о расторжении.
2. Отправить уведомление за 30 дней до желаемой даты...
📊 confidence=0.59 | escalated=False | latency=2613.09ms
---
❓ Сколько стоит парковка?
🤖 Парковка стоит 3500 руб/мес за дополнительное место. Первое место для помещений более 50м² включено бесплатно. Гостевые парковки — бесплатно до 2 часо...
📊 confidence=0.57 | escalated=False | latency=1347.95ms
---
❓ Где покушать?
🤖 Информация не найдена. Ваш вопрос передан оператору. Мы свяжемся с вами в ближайшее время....
📊 confidence=0.44 | escalated=True | latency=214.86ms
---


Проверка версии и API LangFuse

In [13]:
import langfuse
print(f"LangFuse version: {langfuse.__version__}")

# Проверим, какие методы доступны
methods = [m for m in dir(langfuse) if not m.startswith('_')]
print(f"\nДоступные методы Langfuse:")
for m in methods:
    if not m[0].isupper():
        print(f"  - {m}")

LangFuse version: 4.14.1

Доступные методы Langfuse:
  - api
  - batch_evaluation
  - experiment
  - get_client
  - is_default_export_span
  - is_genai_span
  - is_known_llm_instrumentor
  - is_langfuse_span
  - logger
  - media
  - model
  - observe
  - propagate_attributes
  - span_filter
  - types


Трейсинг через @observe

In [14]:
from langfuse import observe

@observe()
def ask_agent(query):
    result = agent.invoke({
        "query": query,
        "chunks": [],
        "answer": "",
        "confidence": 0.0,
        "escalated": False
    })
    return result

# Тест с трейсингом
for q in ["Как расторгнуть договор?", "Сколько стоит парковка?", "Где покушать?"]:
    t0 = time.time()
    result = ask_agent(q)
    latency = round((time.time() - t0) * 1000, 2)

    print(f"❓ {q}")
    print(f"🤖 {result['answer'][:150]}...")
    print(f"📊 confidence={result['confidence']:.2f} | escalated={result['escalated']} | latency={latency}ms")
    print("---")

print("\n✅ Проверь дашборд LangFuse — там должны появиться трейсы!")

❓ Как расторгнуть договор?
🤖 Для расторжения договора аренды необходимо:

1. Подготовить письменное уведомление о расторжении.
2. Отправить уведомление за 30 дней до желаемой даты...
📊 confidence=0.59 | escalated=False | latency=2060.49ms
---
❓ Сколько стоит парковка?
🤖 Парковка стоит 3500 руб/мес за дополнительное место. Первое место для помещений более 50м² включено бесплатно. Гостевые парковки — бесплатно до 2 часо...
📊 confidence=0.57 | escalated=False | latency=1163.28ms
---
❓ Где покушать?
🤖 Информация не найдена. Ваш вопрос передан оператору. Мы свяжемся с вами в ближайшее время....
📊 confidence=0.44 | escalated=True | latency=180.34ms
---

✅ Проверь дашборд LangFuse — там должны появиться трейсы!


Golden set + метрики

In [15]:
from langfuse import observe

golden_set = [
    ("Как расторгнуть договор аренды?", "за 30 дней"),
    ("Сколько стоит дополнительное место на парковке?", "3500"),
    ("Как оплатить аренду?", "до 5 числа"),
    ("Нужна ли страховка помещения?", "обязательно"),
    ("Какой штраф за просрочку аренды?", "0.1%"),
    ("Какой график работы офиса?", "9:00"),
    ("Что проверять при приёмке помещения?", "электрику"),
]

print("📊 ОЦЕНКА АГЕНТА\n")

results = []
for q, expected in golden_set:
    result = ask_agent(q)
    ok = expected.lower() in result["answer"].lower()
    results.append(ok)

    print(f"{'✅' if ok else '❌'} {q}")
    if not ok:
        print(f"   Ожидалось: {expected}")
        print(f"   Ответ: {result['answer'][:100]}...")
    print()

accuracy = sum(results) / len(results)
print(f"📈 Точность: {sum(results)}/{len(results)} ({accuracy*100:.0f}%)")

📊 ОЦЕНКА АГЕНТА

✅ Как расторгнуть договор аренды?

✅ Сколько стоит дополнительное место на парковке?

❌ Как оплатить аренду?
   Ожидалось: до 5 числа
   Ответ: Оплата аренды осуществляется согласно условиям договора. Обычно это делается через банковский перево...

❌ Нужна ли страховка помещения?
   Ожидалось: обязательно
   Ответ: Да, страховка помещения обязательна. Сумма страховки должна быть не менее годовой арендной платы....

✅ Какой штраф за просрочку аренды?

✅ Какой график работы офиса?

✅ Что проверять при приёмке помещения?

📈 Точность: 5/7 (71%)
